In [ ]:
import torch
import numpy as np  
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import pandas as pd
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from src.data.augment import data_transformer
from src.utils.utils import process_dicom_to_3channel
from src.utils.utils import rle2mask
from src.models.classifier import MedicalFusionClassifier
from src.models.segmentor import build_stage2_segmentor
from src.utils.utils import FusionModelWrapper

from configs.configs import get_config
cfg = get_config()

In [ ]:

def plot_error_analysis_suite(df, error_idx, classifier_model, transform_pipeline, device="cuda"):
    """
    Plots a 1x3 grid for FN and FP cases: Original, Grad-CAM, and the Relevant Mask.
    """
    classifier_model.to(device)
    classifier_model.eval()
    print(f"Total error indices to process: {len(error_idx)}")
    
    for count, i in enumerate(error_idx): 
        print(f"Processing item {count+1}/{len(error_idx)} (DataFrame Index: {i})...")
        
        try:
            row = df.loc[i]
            dicom_path = row['path']
            true_label = int(row['class'])
            pred_label = int(row['pred'])
            
            # ---------------------------------------------------------
            # 1. Dynamically Determine Error Type and Target Mask
            # ---------------------------------------------------------
            if true_label == 1 and pred_label == 0:
                error_label = "FALSE NEGATIVE (Patient is Sick, Predicted Healthy)"
                mask_to_show_raw = row['EncodedPixels']
                mask_title = "True Mask (What the model missed)"
                cmap = 'Reds'
            elif true_label == 0 and pred_label == 1:
                error_label = "FALSE POSITIVE (Patient is Healthy, Predicted Sick)"
                mask_to_show_raw = row['pred_mask']
                mask_title = "Predicted Mask (Model Hallucination)"
                cmap = 'Blues'
            else:
                print(f"⚠️ Skipping index {i}: Not an FP or FN.")
                continue

            # ---------------------------------------------------------
            # 2. Base Image Processing (1024x1024)
            # ---------------------------------------------------------
            raw_rgb = process_dicom_to_3channel(dicom_path) 
            scaled_rgb = raw_rgb.astype(np.float32) / 255.0   
            
            # ---------------------------------------------------------
            # 3. Prepare Tensors for the Classifier
            # ---------------------------------------------------------
            meta_values = row[['Age', 'Sex', 'ViewPosition']].astype(float).values
            metadata_tensor = torch.tensor(meta_values, dtype=torch.float32).unsqueeze(0).to(device)
            
            augmented = transform_pipeline(image=raw_rgb)
            image_tensor = augmented['image'].unsqueeze(0).to(device)
            
            # Slice the 3-channel tensor down to 1-channel for your specific model
            image_tensor = image_tensor[:, :1, :, :] 
            
            # ---------------------------------------------------------
            # 4. Generate Grad-CAM Heatmap
            # ---------------------------------------------------------
            wrapped_model = FusionModelWrapper(classifier_model, metadata_tensor)
            target_layers = [classifier_model.image_encoder.conv_head] 
            
            cam = GradCAM(model=wrapped_model, target_layers=target_layers)
            # We target index 0 to see what features pushed the logit towards "Sick"
            targets = [ClassifierOutputTarget(0)] 
            
            grayscale_cam = cam(input_tensor=image_tensor, targets=targets)[0, :]
            grayscale_cam_resized = cv2.resize(grayscale_cam, (raw_rgb.shape[1], raw_rgb.shape[0]))
            heatmap_overlay = show_cam_on_image(scaled_rgb, grayscale_cam_resized, use_rgb=True)

            # ---------------------------------------------------------
            # 5. Mask Decoding & Formatting
            # ---------------------------------------------------------
            # Safely handle empty masks
            if pd.isna(mask_to_show_raw) or str(mask_to_show_raw).strip() == '-1' or str(mask_to_show_raw).strip() == '':
                mask = np.zeros((1024, 1024), dtype=np.float32)
            else:
                mask = rle2mask(mask_to_show_raw, 1024, 1024)
                mask = np.rot90(mask, 3) 
                mask = np.flip(mask, axis=1)
                
            mask_visible = np.ma.masked_where(mask == 0, mask)
            
            # ---------------------------------------------------------
            # 6. Plot the 1x3 Grid
            # ---------------------------------------------------------
            fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
            fig.suptitle(error_label, fontsize=16, fontweight='bold', color='red' if 'NEGATIVE' in error_label else 'orange')
            
            axes[0].imshow(raw_rgb)
            axes[0].set_title("Original Image")
            axes[0].axis('off')
            
            axes[1].imshow(heatmap_overlay)
            axes[1].set_title("Classifier Grad-CAM Focus")
            axes[1].axis('off')
            
            axes[2].imshow(raw_rgb)
            # Only overlay if the mask isn't completely empty
            if np.any(mask):
                axes[2].imshow(mask_visible, cmap=cmap, alpha=0.5, vmin=0, vmax=1) 
            axes[2].set_title(mask_title)
            axes[2].axis('off')  
            
            plt.tight_layout()
            plt.show()
            plt.close(fig) 
            
        except Exception as e:
            print(f"❌ Error at index {i}: {e}")
            continue

In [ ]:

device = torch.device(cfg.device)
classifier_model = MedicalFusionClassifier(
    backbone_name=cfg.model.classifier_backbone, 
    num_meta_features=cfg.model.num_meta_features
).to(device)
classifier_model.load_state_dict(
    torch.load(cfg.model_path.classifier_checkpoint, map_location=device)
)

segmentor_model = build_stage2_segmentor().to(device) # If build_stage2_segmentor takes args, use cfg.model here too
segmentor_model.load_state_dict(
    torch.load(cfg.model_path.segmentor_checkpoint, map_location=device)
)

# 1. Create your validation transform
val_transforms = data_transformer(phase='val', size=1024)
res_df = pd.read_csv('val_data_with_prediction.csv')
# 2. Extract only the errors from the master dataframe
error_df = res_df[((res_df['class'] == 1) & (res_df['pred'] == 0)) | 
                  ((res_df['class'] == 0) & (res_df['pred'] == 1))]

# Select 20 random rows
error_df = error_df.sample(n=15)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 4. Execute the Error Analysis Suite
plot_error_analysis_suite(
    df=res_df,  # Pass the master dataframe
    error_idx=error_df.index,  # Only iterate over the indices where errors occurred
    classifier_model=classifier_model, 
    transform_pipeline=val_transforms,
    device=device
)